# train_reranker_hardneg — reranker-v3: continue-train clf_R on R with dense hard negatives

Continue-trains **from `clf_R`** (not raw BioLinkBERT — v1 regressed by discarding clf's distribution)
on the frozen `elig_first-L512` representation, oversampling the rel=0 judged docs the **dense retriever**
ranks highest per topic — the semantic near-misses a pointwise cross-encoder scores as false positives
(§7i Finding 4). Weak alone; valuable as an ensemble feature (v2 was). Output = reranker_v3.


## Setup (Colab — GPU)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q 'transformers[torch]' datasets accelerate sentence-transformers scikit-learn pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ── ONE-TIME: push clf_R to the Hub, then DELETE this cell ────────────────────────────
# Paste a WRITE token for the semaj83 account between the quotes, run ONCE, then DELETE this
# cell (don't save/share the notebook with the token in it).
import os
os.environ['HF_TOKEN'] = ''   # <-- paste WRITE token here (https://huggingface.co/settings/tokens)
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# _p = '/content/drive/MyDrive/ct_data23/models/clf_R'
# AutoModelForSequenceClassification.from_pretrained(_p).push_to_hub('semaj83/ctmatch-clf-R')
# AutoTokenizer.from_pretrained(_p).push_to_hub('semaj83/ctmatch-clf-R')


In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, torch, torch.nn.functional as F
from ctmatch.experiments import (ExperimentConfig, load_corpus, build_clf_dataset,
                                 mine_dense_hard_negatives, log_result)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = ExperimentConfig(data_root=DATA_ROOT)   # frozen R; cfg.clf_ckpt should be clf_R
print('repr:', cfg.repr_tag(), '| base (continue-train from):', cfg.clf_ckpt)


In [ ]:
# Continue-train config. BASE = clf_R (cfg.clf_ckpt). Low LR / few epochs to avoid forgetting.
BASE_MODEL = cfg.clf_ckpt              # bump cfg.clf_ckpt to clf_R first (or set BASE_MODEL here)
OUT_DIR    = cfg.path('models/reranker_v3')
HUB_REPO   = 'semaj83/ctmatch-reranker-v3'; PUSH = False
N_HARD_NEG, OVERSAMPLE = 50, 3
LR, EPOCHS, BATCH, FOCAL_GAMMA = 5e-6, 2, 8, 2.0
ID2LABEL = {0: 'not_relevant', 1: 'partially_relevant', 2: 'relevant'}


In [ ]:
# Pairs + corpus fields, then mine dense-hard negatives and oversample them into the training set.
train_pairs = [json.loads(l) for l in open(cfg.path('data/clf_pairs_train.jsonl'))]
corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
hard = mine_dense_hard_negatives(cfg, train_pairs, id2fields, n_per_topic=N_HARD_NEG)
aug_pairs = train_pairs + hard * OVERSAMPLE
print(f'base {len(train_pairs):,} + hard {len(hard):,} x{OVERSAMPLE} = {len(aug_pairs):,} training pairs')


In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL).to(device)
model.config.id2label = ID2LABEL; model.config.label2id = {v: k for k, v in ID2LABEL.items()}
train_ds = build_clf_dataset(aug_pairs, id2fields, tokenizer, cfg)
counts = np.bincount([int(p['label']) for p in aug_pairs], minlength=3).astype(float)
w = 1.0 - counts / counts.sum(); class_weights = torch.tensor(w / w.sum(), dtype=torch.float32).to(device)
print('aug label counts:', counts.astype(int))


In [ ]:
class FocalLossTrainer(Trainer):
    def __init__(self, class_weights, gamma=2.0, *a, **k):
        super().__init__(*a, **k); self.class_weights = class_weights; self.gamma = gamma
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels'); out = model(**inputs); logits = out.get('logits')
        ce = F.cross_entropy(logits, labels, weight=self.class_weights, reduction='none')
        loss = ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()
        return (loss, out) if return_outputs else loss

args = TrainingArguments(output_dir=OUT_DIR, learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, warmup_ratio=0.1, weight_decay=0.01, fp16=True,
    save_strategy='epoch', logging_steps=50, report_to='none', seed=cfg.seed)
trainer = FocalLossTrainer(class_weights=class_weights, gamma=FOCAL_GAMMA, model=model, args=args,
    train_dataset=train_ds, data_collator=DataCollatorWithPadding(tokenizer))
trainer.train()


In [ ]:
trainer.save_model(OUT_DIR); tokenizer.save_pretrained(OUT_DIR)
log_result(cfg, experiment='train_reranker_hardneg', split='train',
           metrics={'train_loss': float(trainer.state.log_history[-1].get('train_loss', 0))},
           extra={'base': BASE_MODEL, 'n_hard': len(hard), 'oversample': OVERSAMPLE})
if PUSH: model.push_to_hub(HUB_REPO); tokenizer.push_to_hub(HUB_REPO); print('pushed', HUB_REPO)
print('saved reranker_v3 →', OUT_DIR)
# NOTE: reranker-v3 is weak standalone (like v2); its value is as the v2_rel ENSEMBLE feature.
# Validate it there (train_ensemble_full), not as a standalone reranker.
